# Clase 164 — TD Learning, Q-Learning y DQN (numpy, ejecutable)

**Q-Learning** (Watkins, 1989) es **model-free**, **off-policy** y usa **bootstrap**:
`Q(s,a) ← Q(s,a) + α·[r + γ·max_a' Q(s',a') − Q(s,a)]`.

Implementamos Q-learning **tabular** sobre un gridworld propio en numpy puro (se ejecuta y
converge). Al final mostramos el salto conceptual a **DQN** con Keras (no se ejecuta).

Requiere: `numpy`; `tensorflow` opcional para la parte conceptual de DQN.

## 1. El entorno: gridworld determinista 4×4

```
 0(S)  1     2     3
 4     5(H)  6     7(H)
 8     9    10    11(H)
12(H) 13    14    15(G)
```
Meta `+1`, hueco `-1`, ambos terminales. Determinista para ver la convergencia con claridad.

In [ ]:
import numpy as np
np.random.seed(42)

nS, nA = 16, 4
holes = {5, 7, 11, 12}
goal = 15

def next_state(s, a):
    row, col = divmod(s, 4)
    if   a == 0: col = max(col - 1, 0)
    elif a == 1: row = min(row + 1, 3)
    elif a == 2: col = min(col + 1, 3)
    elif a == 3: row = max(row - 1, 0)
    return row * 4 + col

class GridWorld:
    def reset(self):
        self.s = 0
        return self.s
    def step(self, a):
        s2 = next_state(self.s, a)
        self.s = s2
        if s2 == goal:  return s2,  1.0, True
        if s2 in holes: return s2, -1.0, True
        return s2, 0.0, False

env = GridWorld()
print("GridWorld listo: 16 estados, 4 acciones, meta=15")

## 2. TD error y la actualización de Q-Learning

El **TD error** es `δ = r + γ·max_a' Q(s',a') − Q(s,a)`. Es *off-policy*: el target usa el
`max` (la policy greedy óptima) aunque exploremos con **ε-greedy**.

In [ ]:
def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return rng.integers(nA)              # explora
    return int(np.argmax(Q[s]))              # explota

def q_learning(env, episodes=3000, alpha=0.1, gamma=0.99,
               eps_start=1.0, eps_end=0.01, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((nS, nA))
    for ep in range(episodes):
        eps = max(eps_end, eps_start - ep / (0.7 * episodes))   # decaimiento lineal
        s = env.reset()
        for _ in range(100):
            a = eps_greedy(Q, s, eps, rng)
            s2, r, done = env.step(a)
            target = r + (0.0 if done else gamma * Q[s2].max())
            Q[s, a] += alpha * (target - Q[s, a])               # update TD
            s = s2
            if done:
                break
    return Q

Q = q_learning(env, episodes=3000)
print("Q entrenada. V(s)=max_a Q[s,a]:")
print(np.round(Q.max(axis=1).reshape(4, 4), 3))

## 3. La policy greedy aprendida

In [ ]:
policy = Q.argmax(axis=1)
arrows = np.array(list("<v>^"))
grid = arrows[policy].reshape(4, 4)
for s in holes: grid.flat[s] = "H"
grid.flat[goal] = "G"
print("policy greedy:\n", grid)

## 4. Evaluar: ¿llega a la meta?

Ejecutamos la policy greedy (sin exploración) y medimos el success rate. En el gridworld
determinista una policy óptima alcanza la meta el 100% de las veces.

In [ ]:
def greedy_rollout(env, Q, max_steps=100):
    s = env.reset()
    for _ in range(max_steps):
        s, r, done = env.step(int(np.argmax(Q[s])))
        if done:
            return r > 0            # True si termino en la meta
    return False

success = np.mean([greedy_rollout(env, Q) for _ in range(100)])
print(f"success rate greedy (100 episodios): {success:.2f}")

## 5. De tabular a DQN (conceptual, Keras)

Cuando el espacio de estados es enorme o continuo (píxeles) la tabla `Q[s,a]` no cabe. **DQN**
(Mnih et al., 2015) la reemplaza por una red `s → Q(a)` y añade dos trucos clave:
**experience replay** (romper correlación temporal) y **target network** (estabilizar el target).

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from collections import deque
    TF_OK = True
except Exception:
    TF_OK = False
    print("tensorflow no instalado -> DQN se muestra como referencia (no se ejecuta)")

if TF_OK:
    def build_dqn(n_states=4, n_actions=2):
        return keras.Sequential([
            keras.Input(shape=(n_states,)),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(n_actions),           # Q(s, a) para cada accion (lineal)
        ])
    online = build_dqn()
    target = build_dqn()
    target.set_weights(online.get_weights())         # target network = copia frozen
    replay = deque(maxlen=50_000)                     # experience replay buffer
    # target del batch: y = r + gamma * max_a' Q_target(s', a')   (0 si terminated)
    print("DQN: online + target network + replay buffer listos")
else:
    print("y = r + gamma * max_a' Q_target(s', a')  ; sync target cada N steps")

## Ejercicios

1. Volver el gridworld **resbaladizo** (1/3 intención, 1/3 cada perpendicular) y ver cómo baja
   el success rate y cambia el número de episodios hasta converger.
2. Barrer `α ∈ {0.01, 0.1, 0.5}` y `γ ∈ {0.9, 0.99}`; graficar la curva de aprendizaje.
3. Probar distintas estrategias de decaimiento de `ε` (lineal vs exponencial).
4. Implementar el loop DQN completo sobre `CartPole-v1` con replay buffer y target network sync
   cada 100 steps; entrenar hasta `mean_reward(100) ≥ 195`.

## Conclusiones

- Q-Learning es model-free, off-policy y bootstrapped: aprende `Q*` desde experiencia.
- La actualización TD `Q += α(r + γ·max Q' − Q)` converge sin conocer el modelo del entorno.
- `ε-greedy` equilibra exploración y explotación; el `ε` decae durante el entrenamiento.
- DQN escala Q-learning a estados grandes con una red, experience replay y target network.